# 32 RL: reinforcement learning with an execution reward (A100)

Starts from the SFT model kept on validation (`25_sft_evaluate`). Each iteration takes 128 new train questions (never the validation databases); the model writes 8 answers to each; every answer is executed and scored; then the model is updated to make the better answers of each question more likely than its other answers. Score of an answer: 0 if it is wrong, fails, has no SQL, is cut off, relies on SQLite picking an arbitrary row (`x = (SELECT ...)` with several possible rows, LIMIT without ORDER BY) or uses a constant found nowhere in the question, evidence or gold; a correct answer scores between 0.1 (4x costlier than gold) and 1.9 (4x cheaper), 1 at gold's cost. So a correct answer always beats a wrong one, and among correct answers the cheaper one wins.

1. **Pilot** (2 iterations of 32 questions, about 20 minutes): the table shows the share correct, the mean score and the cost of correct answers per iteration, and the projected time and units of the full run. **Then disconnect** and let me check it before the full run.
2. **Full run** (24 iterations of 128 questions, about 3 hours, about 35 units; measured by the pilot). Copies of the model after 8 and 16 iterations are kept as their own runs.
3. **Predict** validation, dev and mini-dev with the final model and the two copies (a few minutes each): validation then decides how many iterations are best.

Generation (vLLM) and training run in separate processes, one after the other, because both do not fit on a 40 GB A100 together; each iteration loads the model twice (about 2 to 3 minutes). **Resumable:** after a disconnect, run the setup, install and config cells and the staging cell, then the cell that was running: it continues with the first unfinished iteration.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# RL starts from the SFT model kept on validation by R-VES (25_sft_evaluate: method, then epoch 1 or 2).
INIT = json.load(open(f"{R}/{EXPORT}_epoch_choice_rves/selection.json"))["winner"]
PILOT, RUN = "rl_v1_pilot", "rl_v1"
# Snapshots after 8 and 16 of the 24 iterations (own runs), then the final model: compared on validation.
RUNS = [r for r in (f"{RUN}_it8", f"{RUN}_it16") if os.path.exists(f"{R}/{r}/adapter_meta.json")] + [RUN]
print("starts from:", INIT, "| pilot run:", PILOT, "| full run:", RUN)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### Stage the train databases (30 GB, a few minutes; skipped when already staged)

Every answer is executed on its database while training.

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### 1. Pilot (then disconnect and let me check the output)

In [ ]:
!python -m effsql.train rl --run {PILOT} --init-from {INIT} --pilot

### 2. Full run (after the pilot is checked)

In [ ]:
!python -m effsql.train rl --run {RUN} --init-from {INIT}

### 3. Predict validation, dev and mini-dev

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# RL starts from the SFT model kept on validation by R-VES (25_sft_evaluate: method, then epoch 1 or 2).
INIT = json.load(open(f"{R}/{EXPORT}_epoch_choice_rves/selection.json"))["winner"]
PILOT, RUN = "rl_v1_pilot", "rl_v1"
# Snapshots after 8 and 16 of the 24 iterations (own runs), then the final model: compared on validation.
RUNS = [r for r in (f"{RUN}_it8", f"{RUN}_it16") if os.path.exists(f"{R}/{r}/adapter_meta.json")] + [RUN]
for r in RUNS:
    !python -m effsql.train predict --run {r} --splits val,dev,minidev

### Done: disconnect now, then run `33_rl_evaluate` on a CPU runtime